In [51]:
import polars as pl
import os
import glob
from tqdm import tqdm

## Functions

In [52]:
def reorder_cols(df: pl.DataFrame) -> pl.DataFrame:
	cols = df.columns

	if "sample" in cols:
		cols.remove("sample")
		cols.insert(0, "sample")

	return df[cols]

In [53]:
def summarize_res(df: pl.DataFrame) -> pl.DataFrame:
	
	bool_cols = df['filter_1_mutation_intra_hairpin_loop':'filter_8_low_quality'].columns
	str_cols = df['msec_filter_123':'msec_filter_all'].columns
	n_variants = df.height

	# Initialize a dictionary
	results = {
		"filter": [],
		"percentage": [],
		"n_failed": []
	}

	# Populate the dictionary inside the loops
	for col in bool_cols:
		n_failed = df[col].sum() # Sum counts True values
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	for col in str_cols:
		n_failed = df.filter(~pl.col(col).is_null()).height # Count non-nulls
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	# Create DataFrame from the dictionary
	return pl.DataFrame(results).with_columns(pl.lit(n_variants).alias("total_variants"))

### MicroSEC filter Description

The MicroSEC pipeline contains 8 filtering processes.  

- Filter 1  : Shorter-supporting lengths distribute too short to occur (1-1 and 1-2).  
	- Filter 1-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 1-2: The shorter-supporting lengths distributed over less than 75% of the read length.  
- Filter 2  : Hairpin-structure induced error detection (2-1 and 2-2).  
	- Filter 2-1: Palindromic sequences exist within 200 bases.  
	- Filter 2-2: >=50% mutation-supporting reads contains a reverse complementary sequence of the opposite strand consisting >= 15 bases.  
- Filter 3  : 3'-/5'-supporting lengths are too densely distributed to occur (3-1 and 3-2).  
	- Filter 3-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 3-2: The distributions of 3'-/5'-supporting lengths are within 75% of the read length.  
- Filter 4  : >=15% mutations were called by chimeric reads comprising two distant regions.  
- Filter 5  : >=50% mutations were called by soft-clipped reads.  
- Filter 6  : Mutations locating at simple repeat sequences.  
- Filter 7  : Indel mutations locating at a >=15 homopolymer.  
- Filter 8  : >=10% of bases are low quality (Quality score <18) in the mutation supporting reads.  

Filter 1, 2, 3, and 4 detect possible FFPE artifacts.  
Filter 5 may also be FFPE artifacts or mapping errors.  
Filter 6, 7, and 8 detect frequent errors caused by the next generation sequencing platform.  
Supporting lengths are adjusted considering small repeat sequences around the mutations.  
  
Results are saved in a tsv file.  

github url: https://github.com/MANO-B/MicroSEC

## Main
### VCF

In [54]:
msec_paths = sorted(glob.glob("../vcf-micr-svf/*/*.microsec.tsv"))

all_res = []

for path in tqdm(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x : x.lower())
	
	all_res.append(df)
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1180/1180 [00:15<00:00, 74.71it/s] 


In [55]:
final_df

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1623179-01""","""1-ins""","""chr1""",27107272,"""C""","""GT""","""N""","""CGTGGGACACCTCCCCCCCCGTGTGTGTGT…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""1-snv""","""chr1""",45806009,"""C""","""T""","""N""","""CCAGAGGTAGCCTTCAAAGCTTCTGCGCTC…",49,336,4,0,48,48,24,493,251,0.010265,0.009821,0.010417,0.0,0.011905,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""4-del""","""chr1""",65352034,"""AAAAT""","""A""","""N""","""TCAGAGAAGCGCTAAAGACAAAAATAAATA…",49,95,3,0,27,44,27,317,180,0.011386,0.011579,0.008421,0.0,0.031579,0.00001,1.1245e-22,2.7963e-14,false,false,false,false,false,false,false,false,null,null,null,""" filter 3: p is small, but sup…"
"""ORD-1623179-01""","""1-snv""","""chr11""",69458002,"""C""","""T""","""N""","""ACCGACAACTCCATCCGGCCTGAGGAGCTG…",49,302,0,0,48,48,24,315,269,0.012367,0.012583,0.013907,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""1-snv""","""chr11""",108216478,"""A""","""G""","""N""","""CTCTATTTAAAGGAGGTGCAGAAAAAGTCT…",49,354,2,0,48,48,24,355,548,0.010435,0.013559,0.010169,0.0,0.00565,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""1-snv""","""chr17""",37856504,"""G""","""A""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""1-snv""","""chr22""",29108003,"""C""","""T""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""1-snv""","""chr4""",1920021,"""A""","""C""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null


In [56]:
## Samples that could be processed
final_df["sample"].unique()

sample
str
"""ORD-1761543-01"""
"""ORD-1833775-01"""
"""ORD-2092234-01"""
"""ORD-1900033-01"""
"""ORD-1857422-01"""
…
"""ORD-1971886-01"""
"""ORD-2106767-01"""
"""ORD-1988011-01"""


In [57]:
# All Filters
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1623179-01""","""16-del""","""chr14""",95566092,"""CACACACACACACACAC""","""A""","""Y""","""CACACACACACACACACACAAAAACTTACC…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1623179-01""","""14-del""","""chr14""",95566094,"""CACACACACACACAC""","""A""","""Y""","""CACACACACACACACACACAAAAACTTACC…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1623179-01""","""1-snv""","""chr2""",29449221,"""G""","""A""","""N""","""TGGCATGGTGGTGGGCGCCTATAGTCCCAG…",49,21,4,0,48,47,17,315,361,0.165209,0.104762,0.214286,0.47619,0.190476,0.340562,1.0,1.0,false,false,false,true,false,false,false,true,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1623179-01""","""1-snv""","""chr20""",43963375,"""C""","""A""","""N""","""CTGACCTCGTGATCTGCCTGACTTGGCCTC…",49,4,0,0,41,30,18,41,142,0.0,0.0,0.0,1.0,0.0,0.053013,0.043103,0.043103,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1623179-01""","""30-del""","""chr20""",57466734,"""CCCGCCGCCGCCGCAGCCCGGCCGCGCCCC…","""C""","""Y""","""CCCGCGTGAGGCCGCCCGCGCCCGCCGCCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""1-snv""","""chr17""",29483000,"""G""","""T""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""2-ins""","""chr8""",117864264,"""C""","""CAG""","""N""","""TGGTGGAGGCATAGCTGACTCAGATCTATG…",144,37,26,0,118,112,56,145,112,0.010323,0.005405,0.010811,0.0,0.702703,9.2195e-9,0.000591,0.001694,true,false,false,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null


782

In [58]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1623179-01""","""1-snv""","""chr2""",29449221,"""G""","""A""","""N""","""TGGCATGGTGGTGGGCGCCTATAGTCCCAG…",49,21,4,0,48,47,17,315,361,0.165209,0.104762,0.214286,0.47619,0.190476,0.340562,1.0,1.0,false,false,false,true,false,false,false,true,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1623179-01""","""1-snv""","""chr20""",43963375,"""C""","""A""","""N""","""CTGACCTCGTGATCTGCCTGACTTGGCCTC…",49,4,0,0,41,30,18,41,142,0.0,0.0,0.0,1.0,0.0,0.053013,0.043103,0.043103,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1624095-01""","""1-ins""","""chr17""",41204899,"""T""","""TG""","""N""","""TAGGACAAGTCTGTGTGTTTTGTTTTTTTT…",49,47,1,0,35,42,24,297,167,0.074251,0.040426,0.117021,0.0,0.021277,0.000063,1.3711e-14,6.4104e-14,false,false,true,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1624095-01""","""2-ins""","""chr4""",55135671,"""T""","""TAC""","""Y""","""ACTATATTTATGCACATACATACACACACA…",49,31,0,0,8,45,8,218,285,0.016458,0.0,0.0,1.0,0.0,1.0,9.8080e-7,1.7540e-8,false,false,true,true,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1624095-01""","""2-del""","""chr7""",55270842,"""TAC""","""T""","""Y""","""ATATACACACACCACACACATACAGACACC…",49,68,2,0,39,42,26,296,42,0.017107,0.010294,0.017647,0.0,0.029412,2.4527e-12,6.5807e-8,0.000009,true,false,false,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""1-snv""","""chr3""",178952085,"""A""","""G""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""1-snv""","""chr2""",25463568,"""A""","""G""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.010714,0.0125,0.0,0.035714,3.0944e-7,4.7255e-11,9.2555e-11,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"


386

In [59]:
summarize_res(final_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.900257,252,27992
"""filter_2_hairpin_structure""",0.035724,10,27992
"""filter_3_microhomology_induced…",1.303944,365,27992
"""filter_4_highly_homologous_reg…",1.168191,327,27992
"""filter_5_soft_clipped_reads""",0.639468,179,27992
…,…,…,…
"""filter_7_mutation_at_homopolym…",1.989854,557,27992
"""filter_8_low_quality""",3.000857,840,27992
"""msec_filter_123""",1.782652,499,27992


In [60]:
summarize_res(arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",30.694275,252,821
"""filter_2_hairpin_structure""",1.218027,10,821
"""filter_3_microhomology_induced…",44.457978,365,821
"""filter_4_highly_homologous_reg…",39.829476,327,821
"""filter_5_soft_clipped_reads""",5.602923,46,821
…,…,…,…
"""filter_7_mutation_at_homopolym…",10.962241,90,821
"""filter_8_low_quality""",11.81486,97,821
"""msec_filter_123""",60.779537,499,821


In [61]:
summarize_res(arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",9.061489,252,2781
"""filter_2_hairpin_structure""",0.359583,10,2781
"""filter_3_microhomology_induced…",13.124775,365,2781
"""filter_4_highly_homologous_reg…",11.75836,327,2781
"""filter_5_soft_clipped_reads""",6.436534,179,2781
…,…,…,…
"""filter_7_mutation_at_homopolym…",20.028767,557,2781
"""filter_8_low_quality""",30.204962,840,2781
"""msec_filter_123""",17.943186,499,2781


### XML

In [62]:
msec_paths = sorted(glob.glob("../xml-micr-svf/*/*.microsec.tsv"))
len(msec_paths)

1154

In [63]:
all_res = []

for path in tqdm(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x: x.lower())
	
	all_res.append(df)
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1154/1154 [00:14<00:00, 78.05it/s] 


In [64]:
final_df

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1623179-01""","""chr11""",108235819,"""A""","""G""","""ATM""",false,583,"""8861A>G""","""Y2954C""",0.0086,"""missense""","""NM_000051""","""+""",false,"""1-snv""","""N""","""CTCTGTTTAGGTCCTTCTATGTGATCCACT…",49,11,0,0,45,47,19,210,176,0.020408,0.036364,0.027273,0.0,0.0,0.041189,0.418685,0.413847,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""chr11""",118307385,"""C""","""T""","""MLL""",true,187,"""158C>T""","""A53V""",0.5615,"""missense""","""NM_005933""","""+""",false,"""1-snv""","""N""","""CGGCGGTGGCGGCCCCGGGGTGCCCCCCTC…",49,210,2,0,46,48,24,104,490,0.019825,0.025238,0.010476,0.0,0.009524,1.0,0.304281,0.303254,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""chr17""",7577128,"""A""","""C""","""TP53""",false,631,"""810T>G""","""F270L""",0.2377,"""missense""","""NM_000546""","""-""",false,"""1-snv""","""N""","""CAGGCACAAACACGCACCTCCAAGCTGTTC…",49,352,1,0,48,48,24,379,351,0.013219,0.012216,0.013352,0.0,0.002841,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""chr17""",29550488,"""A""","""G""","""NF1""",false,532,"""1748A>G""","""K583R""",0.1861,"""missense""","""NM_001042492""","""+""",false,"""1-snv""","""N""","""AATGCTTTTTTACATCTGCAGGAAATTAAC…",49,157,0,0,47,48,24,413,508,0.008969,0.011465,0.005732,0.0,0.0,1.0,0.144419,0.144419,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1623179-01""","""chr20""",36012792,"""C""","""T""","""SRC""",true,210,"""236C>T""","""A79V""",0.4905,"""missense""","""NM_005417""","""+""",false,"""1-snv""","""N""","""CGTCACCTCCCCGCAGAGGGTGGGCCCGCT…",49,180,2,0,48,48,24,198,230,0.004535,0.005556,0.005556,0.0,0.011111,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""chr17""",37856504,"""G""","""A""","""ERBB2""",true,3682,"""13G>A""","""A5T""",0.0019,"""missense""","""NM_004448""","""+""",false,"""1-snv""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""chr22""",29108003,"""C""","""T""","""CHEK2""",true,3729,"""686G>A""","""G229D""",0.0013,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""chr4""",1920021,"""A""","""C""","""WHSC1""",true,1979,"""1081A>C""","""K361Q""",0.5073,"""missense""","""NM_133335""","""+""",false,"""1-snv""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,false,false,false,fals

In [65]:
## Samples that could be processed
final_df["sample"].unique()

sample
str
"""ORD-1945290-01"""
"""ORD-1985430-01"""
"""ORD-1994335-01"""
"""ORD-1903615-01"""
"""ORD-1995114-01"""
…
"""ORD-1770600-01"""
"""ORD-1627855-01"""
"""ORD-1635966-01"""


In [66]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1624061-01""","""chr5""",149439360,"""C""","""G""","""CSF1R""",true,1739,"""2035G>C""","""A679P""",0.0219,"""missense""","""NM_005211""","""-""",false,"""1-snv""","""N""","""GGGTCCCAGCATGGCCTCAGGCTTCCTTCG…",144,298,200,0,138,143,70,294,231,0.011722,0.015101,0.013087,0.0,0.671141,0.059965,0.009989,1.0,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1624095-01""","""chr5""",1295250,"""G""","""A""","""TERT""",false,118,"""-146C>T""","""promoter -146C>T""",0.1271,"""promoter""","""NM_198253""","""-""",false,"""1-snv""","""N""","""GGGCTGGGCCGGGGACCCGGAAGGGGTCGG…",49,20,9,0,46,43,23,185,170,0.082653,0.005,0.175,0.0,0.45,0.180786,0.079611,0.054683,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1624922-01""","""chr5""",79950719,"""C""","""T""","""MSH3""",true,376,"""173C>T""","""A58V""",0.5452,"""missense""","""NM_002439""","""+""",false,"""1-snv""","""Y""","""TGCAGCGGCTGCAGCGGCCGTAGCGGCCGC…",144,2200,411,0,143,142,71,456,279,0.085773,0.028864,0.1385,0.0,0.186818,1.0,0.220272,0.220272,false,false,false,false,false,true,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1625835-01""","""chr19""",13054620,"""G""","""C""","""CALR""",true,843,"""1147G>C""","""E383Q""",0.331,"""missense""","""NM_004343""","""+""",false,"""1-snv""","""Y""","""GCAAAGAGGAGGAGGAGGCACAGGACAAGG…",49,462,0,0,48,48,24,411,306,0.013075,0.016017,0.012987,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1625835-01""","""chr9""",98278972,"""T""","""C""","""PTCH1""",false,893,"""131A>G""","""E44G""",0.5812,"""missense""","""NM_001083603""","""-""",false,"""1-snv""","""Y""","""CTCCGTTTTCTTCTTCTTCTCCTCCTCCTC…",49,898,4,0,48,48,24,316,351,0.015977,0.019042,0.016147,0.0,0.004454,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""chr17""",29483000,"""G""","""T""","""NF1""",false,2006,"""61-1G>T""","""splice site 61-1G>T""",0.0957,"""splice""","""NM_001042492""","""+""",false,"""1-snv""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""CUL3""",false,1027,"""67-1_81delGATGACCATGGATGAA""","""splice site 67-1_81delGATGACCA…",0.0351,"""splice""","""NM_003590""","""-""",false,"""16-del""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""chr8""",117864264,"""C""","""CAG""

565

In [67]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1629818-01""","""chr8""",11606569,"""C""","""T""","""GATA4""",true,1516,"""758C>T""","""P253L""",0.0033,"""missense""","""NM_002052""","""+""",false,"""1-snv""","""N""","""GATGAACGGCATCAACCGGCTGCTCATCAA…",144,41,1,0,116,86,68,179,86,0.011348,0.004878,0.017073,0.0,0.02439,4.3110e-7,2.3076e-10,4.2611e-10,true,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1632705-01""","""chr6""",33287880,"""GCCT""","""G""","""DAXX""",true,309,"""1370_1372delAGG""","""E457del""",0.5534,"""nonframeshift""","""NM_001350""","""-""",false,"""3-del""","""Y""","""CCTCCTCTTCAGAATCTGTGGCCTCCTCTT…",49,156,0,0,17,48,17,248,300,0.010204,0.007051,0.00641,0.0,0.0,6.8411e-36,7.6504e-67,4.6507e-71,true,false,true,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1637956-01""","""chr6""",117622150,"""ATTTATG""","""A""","""ROS1""",true,767,"""6714_6719delCATAAA""","""I2239_N2240del""",0.086,"""nonframeshift""","""NM_002944""","""-""",false,"""6-del""","""N""","""AACTTACCTTCAAAGCTTTCAACTCCACTG…",49,73,7,0,36,42,28,224,260,0.002516,0.00274,0.00274,0.0,0.09589,5.3411e-13,1.5029e-12,1.3808e-11,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1641827-01""","""chr2""",61147581,"""C""","""T""","""REL""",true,241,"""986C>T""","""P329L""",0.5228,"""missense""","""NM_002908""","""+""",false,"""1-snv""","""N""","""CCTGACATCAGGTGATCCACTCACCTTGGC…",144,1146,62,0,138,143,71,575,567,0.014774,0.019459,0.011344,0.56719,0.054101,1.0,0.136989,0.136989,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1641912-01""","""chr2""",25463182,"""G""","""A""","""DNMT3A""",false,1373,"""2311C>T""","""R771*""",0.0036,"""nonsense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""TGGCTATACCTCGAGAAATCACGAGATGTC…",144,33,0,0,109,129,47,109,299,0.010101,0.012121,0.0,0.0,0.0,5.1456e-11,0.000006,0.000007,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""chr3""",178952085,"""A""","""G""","""PIK3CA""",false,703,"""3140A>G""","""H1047R""",0.01,"""missense""","""NM_006218""","""+""",false,"""1-snv""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""chr2""",25463568,"""A""","""G""","""DNMT3A""",true,2024,"""2114T>C""","""I705T""",0.004,"""missense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.

171

In [68]:
summarize_res(final_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.658273,103,15647
"""filter_2_hairpin_structure""",0.0,0,15647
"""filter_3_microhomology_induced…",0.6391,100,15647
"""filter_4_highly_homologous_reg…",0.249249,39,15647
"""filter_5_soft_clipped_reads""",0.818048,128,15647
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.006391,1,15647
"""filter_8_low_quality""",2.019556,316,15647
"""msec_filter_123""",1.035342,162,15647


In [69]:
summarize_res(arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",51.243781,103,201
"""filter_2_hairpin_structure""",0.0,0,201
"""filter_3_microhomology_induced…",49.751244,100,201
"""filter_4_highly_homologous_reg…",19.402985,39,201
"""filter_5_soft_clipped_reads""",8.457711,17,201
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.0,0,201
"""filter_8_low_quality""",3.9801,8,201
"""msec_filter_123""",80.597015,162,201


In [70]:
summarize_res(arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",11.381215,103,905
"""filter_2_hairpin_structure""",0.0,0,905
"""filter_3_microhomology_induced…",11.049724,100,905
"""filter_4_highly_homologous_reg…",4.309392,39,905
"""filter_5_soft_clipped_reads""",14.143646,128,905
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.110497,1,905
"""filter_8_low_quality""",34.917127,316,905
"""msec_filter_123""",17.900552,162,905
